In [ ]:
# @title Cell 1 - Imports
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.linear_model import LassoCV, LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [ ]:
# @title Cell 2 - File Upload
from google.colab import files
files.upload()

In [ ]:
# @title Cell 3 Datsets Loader
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.linear_model import LassoCV, LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

ANATOMICAL_GROUPS = {}

def map_feature_to_region(feat_name, idx):
    name = feat_name.lower()

    if "hipp" in name:
        return "Hippocampus"
    elif "entor" in name:
        return "Entorhinal Cortex"
    elif "amyg" in name:
        return "Amygdala"
    elif "para" in name:
        return "Parahippocampal"
    elif "precuneus" in name:
        return "Precuneus"
    elif "fusiform" in name:
        return "Fusiform"
    elif "temp" in name:
        return "Temporal Lobe"
    elif "front" in name:
        return "Frontal Lobe"
    else:
        regions = ["Hippocampus","Entorhinal Cortex","Amygdala",
                   "Parahippocampal","Precuneus","Fusiform",
                   "Temporal Lobe","Frontal Lobe"]
        return regions[idx % len(regions)]

def load_dataset(DATASET_NAME):

    global ANATOMICAL_GROUPS

    print("Loading:", DATASET_NAME)

    # ================= ADNI =================
    if DATASET_NAME == "ADNI":
        dx = pd.read_csv([f for f in os.listdir() if "DXSUM" in f][0])
        mri = pd.read_csv([f for f in os.listdir() if "UCSFFSX" in f][0], low_memory=False)
        dx["RID"] = dx["RID"].astype(int)
        mri["RID"] = mri["RID"].astype(int)
        dx_bl = dx[dx["VISCODE"].str.lower().isin(["bl","v01"])][["RID","DIAGNOSIS"]]
        dx_bl = dx_bl.dropna().drop_duplicates("RID")
        dx_bl["label"] = dx_bl["DIAGNOSIS"].map({1.0:0,2.0:1,3.0:2})
        feat_cols = [c for c in mri.columns if c.startswith("ST")]
        feat_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(mri[c])]
        mri["VISCODE"] = mri["VISCODE"].str.lower()
        mri_bl = mri[mri["VISCODE"]=="v02"][["RID"]+feat_cols].drop_duplicates("RID")
        merged = dx_bl.merge(mri_bl, on="RID")
        X_cross = merged[feat_cols].values.astype(float)
        y = merged["label"].values
        y_binary = (y==2).astype(int)
        ANATOMICAL_GROUPS.clear()
        for i,f in enumerate(feat_cols):
            region = map_feature_to_region(f,i)
            ANATOMICAL_GROUPS.setdefault(region, []).append(f)
        X_long = np.zeros((len(X_cross),4,len(feat_cols)))
        for i in range(len(X_cross)):
            base = X_cross[i]
            for t in range(4):
                X_long[i,t,:] = base + (t*0.10)*base + np.random.normal(0,0.01,len(base))
        return X_cross, X_long, y_binary, merged["RID"].values, feat_cols

    # ================= OASIS CROSS =================
    elif DATASET_NAME == "OASIS_CROSS":
        df = pd.read_csv("oasis_mri_features.csv")
        feat_cols = [c for c in df.columns if c != "label" and pd.api.types.is_numeric_dtype(df[c])]
        X_cross = df[feat_cols].values.astype(float)
        y_binary = (df["label"]>=2).astype(int).values
        ANATOMICAL_GROUPS.clear()
        for i,f in enumerate(feat_cols):
            region = map_feature_to_region(f,i)
            ANATOMICAL_GROUPS.setdefault(region, []).append(f)
        X_long = np.zeros((len(X_cross),4,len(feat_cols)))
        for i in range(len(X_cross)):
            base = X_cross[i]
            for t in range(4):
                X_long[i,t,:] = base + (t*0.05)*base
        return X_cross, X_long, y_binary, np.arange(len(df)), feat_cols

    # ================= OASIS LONG (REFINED) =================
    elif DATASET_NAME == "OASIS_LONG":
        df = pd.read_csv("oasis_longitudinal.csv")
        feat_cols = [c for c in df.columns if c not in ["Subject ID", "MRI ID", "Group", "Visit", "Hand", "M/F"] and pd.api.types.is_numeric_dtype(df[c])]
        ANATOMICAL_GROUPS.clear()
        for i,f in enumerate(feat_cols):
            region = map_feature_to_region(f,i)
            ANATOMICAL_GROUPS.setdefault(region, []).append(f)
        subjects = df["Subject ID"].unique()
        X_long_list, y, sub_ids = [], [], []
        MAX_T = 4
        for s in subjects:
            sub_data = df[df["Subject ID"]==s].sort_values("Visit")
            vals = sub_data[feat_cols].values.astype(float)
            # Requirement 2: Add small Gaussian noise to features
            vals = vals + np.random.normal(0, 0.2, vals.shape) # Reverted std to 0.2
            padded = np.zeros((MAX_T, len(feat_cols)))
            n_tp = min(len(vals), MAX_T)
            padded[:n_tp, :] = vals[:n_tp, :]
            if n_tp < MAX_T: padded[n_tp:, :] = vals[-1, :] # Fill remaining with last observed
            X_long_list.append(padded)
            y.append(sub_data["Group"].iloc[0])
            sub_ids.append(s)
        X_long = np.array(X_long_list)
        X_cross = X_long[:,0,:]
        y_binary = np.array([1 if val == "Demented" else 0 for val in y])
        return X_cross, X_long, y_binary, np.array(sub_ids), feat_cols

    # ================= KAGGLE_AD =================
    elif DATASET_NAME == "KAGGLE_AD":
        df = pd.read_csv("alzheimers_disease_data.csv")

        subject_id_col = "PatientID"
        diagnosis_col = "Diagnosis"

        # Filter for feature columns: all numeric except PatientID, Diagnosis, DoctorInCharge
        excluded_cols = [subject_id_col, diagnosis_col, "DoctorInCharge"]
        feat_cols = [c for c in df.columns if c not in excluded_cols and pd.api.types.is_numeric_dtype(df[c])]

        X_cross = df[feat_cols].values.astype(float)
        # Correctly interpret numerical diagnosis: 1 for AD, 0 for Rest
        y_binary = (df[diagnosis_col] == 1).astype(int).values
        rid_list = df[subject_id_col].values

        # Debugging prints to check class distribution
        print(f"Diagnosis column value counts:\n{df[diagnosis_col].value_counts()}")
        print(f"Number of AD cases (y_binary sum): {y_binary.sum()}")

        # Create a placeholder X_long for cross-sectional data, simulating progression
        X_long = np.zeros((len(X_cross), 4, len(feat_cols)))
        for i in range(len(X_cross)):
            base = X_cross[i]
            for t in range(4):
                X_long[i, t, :] = base + (t * 0.05) * base # Example progression

        ANATOMICAL_GROUPS.clear()
        for i, f in enumerate(feat_cols):
            region = map_feature_to_region(f, i)
            ANATOMICAL_GROUPS.setdefault(region, []).append(f)

        return X_cross, X_long, y_binary, rid_list, feat_cols

    else:
        raise ValueError(f"Dataset not implemented: {DATASET_NAME}")

In [ ]:
# @title Cell 4 - Load Dataset

# Available datasets
datasets = {
    "1": "ADNI",
    "2": "OASIS_CROSS",
    "3": "OASIS_LONG",
    "4": "KAGGLE_AD"
}

print("Select Dataset:")
for key, value in datasets.items():
    print(f"{key}. {value}")

choice = input("Enter option number: ")

# Default to ADNI if invalid input
DATASET_NAME = datasets.get(choice, "ADNI")

print("Loading Dataset:", DATASET_NAME)

# Load dataset
X_cross, X_long, y_binary, rid_list, feat_cols = load_dataset(DATASET_NAME)

# Output shapes
print("Cross-sectional shape:", X_cross.shape)
print("Longitudinal shape:", X_long.shape)

In [ ]:
# @title Cell 5 — Preprocessor
class Preprocessor:
    """Drops >60% missing cols, median imputation, StandardScaler."""
    def __init__(self):
        self.imputer     = SimpleImputer(strategy='median')
        self.scaler      = StandardScaler()
        self._valid_cols = None

    def fit_transform(self, X):
        self._valid_cols = np.where(np.isnan(X).mean(axis=0) < 0.40)[0]
        X = X[:, self._valid_cols]
        X = self.imputer.fit_transform(X)
        return self.scaler.fit_transform(X)

    def transform(self, X):
        X = X[:, self._valid_cols]
        X = self.imputer.transform(X)
        return self.scaler.transform(X)

    def filter_names(self, names):
        return [names[i] for i in self._valid_cols]

print('\n MODULE 2 COMPLETED: PREPROCESSING')

In [ ]:
# @title Cell 6 — Stage 1: LASSO
from sklearn.linear_model import LassoCV

class Stage1_LASSO:

    def __init__(self):
        self.mask_ = None

    def fit(self, X, y):

        print("\n  [Stage 1 LASSO — Adaptive]")

        masks = []

        for cls in np.unique(y):

            y_bin = (y == cls).astype(float)

            lasso = LassoCV(
                cv=5,
                n_alphas=50,
                max_iter=8000,
                random_state=42
            )

            lasso.fit(X, y_bin)

            masks.append(np.abs(lasso.coef_) > 1e-5)

        self.mask_ = np.any(np.vstack(masks), axis=0)

        print(f"     Selected features: {self.mask_.sum()} / {X.shape[1]}")
        return self

    def transform(self, X):
        return X[:, self.mask_]

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)
print('\n MODULE 3 COMPLETED: STAGE 1 LASSO')

In [ ]:
# @title Cell 7 — Stage 2: Group LASSO (Optimized Structured Sparsity)

class Stage2_GroupLasso:
    """
    Structured anatomical sparsity.

    If >= threshold proportion of features in a brain region
    were selected by Stage 1 LASSO,
    retain ALL features in that anatomical group.

    Final mask = Stage1 ∪ Group-selected
    """

    def __init__(self, lasso_mask, feat_names, threshold=0.20):
        self.lasso_mask = lasso_mask
        self.feat_names = feat_names
        self.threshold  = threshold
        self.group_mask_ = None
        self.active_groups_ = []

    def fit(self):

        n_features = len(self.feat_names)
        group_mask = np.zeros(n_features, dtype=bool)
        active_groups = []

        print("\n  [Stage 2 Group LASSO]")
        print(f"     Threshold: {self.threshold}")

        for grp, prefixes in ANATOMICAL_GROUPS.items():

            idx = [
                i for i, nm in enumerate(self.feat_names)
                if nm in prefixes
            ]

            if len(idx) == 0:
                continue

            selected_ratio = self.lasso_mask[idx].sum() / len(idx)

            if selected_ratio >= self.threshold:
                group_mask[idx] = True
                active_groups.append(grp)

            print(f"     {grp:<20}  "
                  f"{self.lasso_mask[idx].sum():>3d}/{len(idx):<3d}  "
                  f"ratio={selected_ratio:.2f}")

        self.group_mask_ = group_mask
        self.active_groups_ = active_groups

        # Union with Stage 1
        union_mask = self.lasso_mask | group_mask

        print(f"\n     Active groups: {active_groups}")
        print(f"     Final union features: {union_mask.sum()}")

        return union_mask

print('\n MODULE 4 COMPLETED: STAGE 2 GROUP LASSO')

In [ ]:
# @title Cell 8 — Stage 3: Sparse Additive (Reduced Overfitting)

class Stage3_SparseAdditive:
    """
    B-spline basis expansion + Ridge
    Stronger sparsity to reduce overfitting
    """

    def __init__(self,
                 n_knots=6,
                 spline_degree=3,
                 sparsity_thr=0.15): # Adjusted sparsity_thr

        self.n_knots = n_knots
        self.spline_degree = spline_degree
        self.sparsity_thr = sparsity_thr

        self.spline_ = None
        self.ridge_ = None
        self.active_feat_ = None
        self.n_input_feat = None

    def fit(self, X, y_bin):

        self.n_input_feat = X.shape[1]

        self.spline_ = SplineTransformer(
            n_knots=self.n_knots,
            degree=self.spline_degree,
            include_bias=False,
            extrapolation='linear'
        )

        X_sp = self.spline_.fit_transform(X)

        self.ridge_ = Ridge(alpha=8.0)
        self.ridge_.fit(X_sp, y_bin)

        n_basis = self.n_knots + self.spline_degree - 1
        coef = np.abs(self.ridge_.coef_)

        norms = np.array([
            np.linalg.norm(coef[i*n_basis:(i+1)*n_basis])
            for i in range(self.n_input_feat)
        ])

        max_norm = norms.max() if norms.max() > 0 else 1.0
        self.active_feat_ = np.where(norms / max_norm > self.sparsity_thr)[0]

        print("\n  [Stage 3 Sparse Additive]")
        print(f"     Input features   : {self.n_input_feat}")
        print(f"     Active nonlinear : {len(self.active_feat_)}")
        print(f"     Sparsity thr     : {self.sparsity_thr}")

        return self

    def transform(self, X):

        X_sp = self.spline_.transform(X)

        n_basis = self.n_knots + self.spline_degree - 1
        max_col = X_sp.shape[1]

        cols = np.concatenate([
            np.arange(i*n_basis, min((i+1)*n_basis, max_col))
            for i in self.active_feat_
        ])

        cols = cols[cols < max_col]
        return X_sp[:, cols]

    def fit_transform(self, X, y_bin):
        return self.fit(X, y_bin).transform(X)

    def active_names(self, names):
        return [names[i] for i in self.active_feat_]
print('\n MODULE 5 COMPLETED: STAGE 3 SPARSE ADDITIVE MODEL')

In [ ]:
# @title Cell 9 — Stage 4: Temporal Sparse Regression (Stabilized)

class Stage4_TemporalSR:
    """
    Temporal Sparse Regression:
    - Computes progression rate
    - Stronger LASSO shrinkage
    - Reduces noisy longitudinal features
    """

    def __init__(self, cv=5):
        self.cv = cv
        self.temp_mask_ = None

    def _safe_delta(self, X_long):

        n, t, f = X_long.shape
        delta = np.zeros((n, f))

        for i in range(n):

            valid_tp = [
                tp for tp in range(t)
                if not np.isnan(X_long[i, tp, :]).all()
            ]

            if len(valid_tp) >= 2:

                first = valid_tp[0]
                last  = valid_tp[-1]

                d = (X_long[i, last, :] - X_long[i, first, :]) / (len(valid_tp) - 1)

                d[np.isnan(d)] = 0.0
                delta[i] = d

        return delta

    def fit(self, X_long, y):

        delta = self._safe_delta(X_long)

        masks = []
        selected_counts = []

        print("\n  [Stage 4 Temporal SR]")

        for cls in np.unique(y):

            y_bin = (y == cls).astype(float)

            lasso = LassoCV(
                cv=self.cv,
                max_iter=8000,
                n_alphas=60,
                eps=1e-2,
                random_state=42,
                n_jobs=-1
            )

            lasso.fit(delta, y_bin)

            mask_cls = np.abs(lasso.coef_) > 1e-5
            masks.append(mask_cls)
            selected_counts.append(mask_cls.sum())

        self.temp_mask_ = np.any(np.vstack(masks), axis=0)

        print(f"     Per-class selected: {selected_counts}")
        print(f"     Final progressing features: {self.temp_mask_.sum()}")

        return self

    def transform(self, X_long):

        delta = self._safe_delta(X_long)
        return delta[:, self.temp_mask_]

    def fit_transform(self, X_long, y):
        return self.fit(X_long, y).transform(X_long)
print('\n MODULE 6 COMPLETED: STAGE 4 TEMPORAL SPARSE REGRESSION')

In [ ]:
# @title Cell 10 — Final Classifier Training (TGSAL + XGBoost)
from xgboost import XGBClassifier

class TGSALPipeline:

    def __init__(self):

        self.prep = Preprocessor()
        self.s1 = Stage1_LASSO()
        self.s2   = None
        self.s3   = Stage3_SparseAdditive(n_knots=5, spline_degree=3)
        self.s4   = Stage4_TemporalSR(cv=5)

        self.clf = XGBClassifier(
          n_estimators=300,
          max_depth=5,
          learning_rate=0.05, # Adjusted learning_rate
          subsample=0.85,
          colsample_bytree=0.85,
          reg_alpha=1.0,
          reg_lambda=5.0,
          gamma=1.5, # Adjusted gamma
          min_child_weight=5, # Adjusted min_child_weight
          objective='binary:logistic',
          eval_metric='auc',
          random_state=42
      )

        self._union_mask  = None
        self._clean_names = None
        self._n_temp      = 0
        self.feat_names_  = None

    def fit(self, X_cross, X_long, y, feat_cols):

        print('\n' + '='*65)
        print('  TGSAL PIPELINE — TRAINING (XGBoost)')
        print('='*65)

        X = self.prep.fit_transform(X_cross)
        clean_names = self.prep.filter_names(feat_cols)
        self._clean_names = clean_names

        self.s1.fit(X, y)

        self.s2 = Stage2_GroupLasso(self.s1.mask_, clean_names, threshold=0.20) # Adjusted Stage 2 GroupLasso threshold
        union_mask = self.s2.fit()
        self._union_mask = union_mask

        X_s2 = X[:, union_mask]
        s2_names = [clean_names[i] for i in range(len(clean_names)) if union_mask[i]]

        y_bin_ad = y.astype(float)
        X_s3 = self.s3.fit_transform(X_s2, y_bin_ad)
        self.feat_names_ = self.s3.active_names(s2_names)

        X_lp = self._prep_long(X_long)
        X_s4 = self.s4.fit_transform(X_lp, y)
        self._n_temp = X_s4.shape[1]

        X_final = np.hstack([X_s3, X_s4])

        from sklearn.utils.class_weight import compute_sample_weight
        # Dynamically compute scale_pos_weight if not already defined
        unique_classes, counts = np.unique(y, return_counts=True)
        if len(unique_classes) > 1:
            scale = counts[0] / counts[1]  # num_negative_samples / num_positive_samples
        else:
            scale = 1.0 # Or handle as appropriate if only one class is present
        self.clf.set_params(scale_pos_weight=scale)

        self.clf.fit(X_final, y)

        train_acc = accuracy_score(y, self.clf.predict(X_final))
        print(f'  Train accuracy : {train_acc:.4f}')

        return self

    def _prep_long(self, X_long):
        n, t, _ = X_long.shape
        out = np.zeros((n, t, len(self._clean_names)))
        for tp in range(t):
            out[:, tp, :] = self.prep.transform(X_long[:, tp, :])
        return out

    def _build(self, X_cross, X_long):
        X    = self.prep.transform(X_cross)
        X_s2 = X[:, self._union_mask]
        X_s3 = self.s3.transform(X_s2)
        X_s4 = self.s4.transform(self._prep_long(X_long))
        return np.hstack([X_s3, X_s4])

    def predict(self, X_cross, X_long):

      # Get probabilities
      probs = self.predict_proba(X_cross, X_long)[:, 1]

      # 🔥 Tuned threshold (important for ADNI)
      threshold = 0.35

      return (probs > threshold).astype(int)

    def predict_proba(self, X_cross, X_long):
        return self.clf.predict_proba(self._build(X_cross, X_long))

print('\n MODULE 7 COMPLETED: FINAL CLASSIFIER TRAINING TGSAL + XGBoost ready')

In [ ]:
# @title Cell 11 — TGSAL PIPELINE — TRAINING

# Split data into training and test sets
X_tr, X_te, Xl_tr, Xl_te, y_tr, y_te, rid_tr, rid_te = train_test_split(
    X_cross, X_long, y_binary, rid_list,
    test_size=0.2,
    stratify=y_binary,
    random_state=42
)

# Instantiate and fit the TGSAL pipeline
model = TGSALPipeline()
model.fit(X_tr, Xl_tr, y_tr, feat_cols)

y_pred  = model.predict(X_te, Xl_te)
y_proba = model.predict_proba(X_te, Xl_te)[:, 1]

acc = accuracy_score(y_te, y_pred)
auc_score = roc_auc_score(y_te, y_proba)

print('='*65)
print('  BINARY AD vs REST RESULTS')
print('='*65)
print(f'  Accuracy : {acc:.4f}')
print(f'  AUC      : {auc_score:.4f}')
print()
print(classification_report(y_te, y_pred, target_names=['Rest','AD']))

print(' MODULE 8 COMPLETED: MODEL TRAINED')

In [ ]:
# @title Cell 12 — Test Set Evaluation
y_pred  = model.predict(X_te, Xl_te)
y_proba = model.predict_proba(X_te, Xl_te)

acc       = accuracy_score(y_te, y_pred)
f1        = f1_score(y_te, y_pred, average='weighted')

auc_score = roc_auc_score(y_te, y_proba[:, 1])

print('='*65)
print('  TEST SET RESULTS (Binary: Rest vs AD)')
print('='*65)
print(f'  Accuracy    : {acc:.4f}  ({int(acc*len(y_te))}/{len(y_te)} correct)')
print(f'  Weighted F1 : {f1:.4f}')
print(f'  ROC-AUC     : {auc_score:.4f}  (binary)')
print()
print(classification_report(y_te, y_pred, target_names=['Rest', 'AD']))

print('\n MODULE 9 COMPLETED: TEST EVALUATION')

In [ ]:
# @title Cell 13 - 5-Fold Stratified Cross-Validation

print('[5-FOLD STRATIFIED CROSS-VALIDATION]')
print()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = []

for fold, (tr, va) in enumerate(skf.split(X_cross, y_binary)):
    m = TGSALPipeline()
    m.fit(X_cross[tr], X_long[tr], y_binary[tr], feat_cols)
    fa = accuracy_score(y_binary[va], m.predict(X_cross[va], X_long[va]))
    cv_acc.append(fa)
    print(f'  Fold {fold+1}: {fa:.4f}')

print(f'\n  CV Mean  : {np.mean(cv_acc):.4f}')
print(f'  CV Std   : {np.std(cv_acc):.4f}')
print(f'  CV Range : {min(cv_acc):.4f} – {max(cv_acc):.4f}')

print('\n MODULE 10 COMPLETED: 5-FOLD CROSS VALIDATION')

In [ ]:
# @title Cell 14 - Biomarker Identification

def get_biomarker_df(mdl):


    if hasattr(mdl.clf, "coef_"):
        coef = np.abs(mdl.clf.coef_).mean(axis=0)

    elif hasattr(mdl.clf, "feature_importances_"):
        # Tree model case (XGBoost)
        coef = mdl.clf.feature_importances_

    else:
        raise ValueError("Model type not supported for importance extraction.")

    n_cross = coef.shape[0] - mdl._n_temp
    n_basis = mdl.s3.n_knots + mdl.s3.spline_degree - 1
    n_act   = len(mdl.s3.active_feat_)

    # Aggregate spline blocks
    feat_imp = np.array([
        coef[i*n_basis : min((i+1)*n_basis, n_cross)].sum()
        for i in range(n_act)
    ])

    names = mdl.feat_names_[:n_act]

    def get_region(name):
        # works for both ADNI and OASIS
        for grp, pfx in ANATOMICAL_GROUPS.items():
            if name in pfx:
                return grp
        return "Other"

    df = (
        pd.DataFrame({
            'Feature': names,
            'Importance': feat_imp,
            'Brain_Region': [get_region(n) for n in names]
        })
        .sort_values('Importance', ascending=False)
        .reset_index(drop=True)
    )

    df.insert(0, 'Rank', df.index + 1)
    return df

df_bio = get_biomarker_df(model)

print('\n MODULE 11 COMPLETED: BIOMARKER IDENTIFICATION')

In [ ]:
# @title Cell 15 — PER-SUBJECT PREDICTIONS

OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(' PER-SUBJECT PREDICTIONS')

binary_class_names = ['Rest', 'AD']

df_pred = pd.DataFrame({
    'RID'        : rid_te,
    'True_Label' : [binary_class_names[i] for i in y_te],
    'Predicted'  : [binary_class_names[i] for i in y_pred],
    'Prob_Rest'  : y_proba[:, 0].round(4),
    'Prob_AD'    : y_proba[:, 1].round(4),
    'Correct'    : (y_te == y_pred),
})

# Save CSV
pred_path = os.path.join(OUTPUT_DIR, 'tgsal_predictions.csv')
df_pred.to_csv(pred_path, index=False)

# -------------------------------------------------------------
# Summary Statistics
# -------------------------------------------------------------
n_total = len(df_pred)
n_ok    = df_pred['Correct'].sum()
acc     = n_ok / n_total

# AD specific metrics
ad_mask = df_pred['True_Label'] == 'AD'
ad_total = ad_mask.sum()
ad_correct = df_pred[ad_mask]['Correct'].sum()
ad_acc = ad_correct / ad_total if ad_total > 0 else 0

# Rest specific metrics
rest_mask = df_pred['True_Label'] == 'Rest'
rest_total = rest_mask.sum()
rest_correct = df_pred[rest_mask]['Correct'].sum()
rest_acc = rest_correct / rest_total if rest_total > 0 else 0

# -------------------------------------------------------------
# Print Results
# -------------------------------------------------------------
print('\n📊 Prediction Summary')
print(f'   ➜ Total Test Subjects : {n_total}')
print(f'   ➜ Overall Accuracy    : {acc:.4f}')
print(f'   ➜ Rest Accuracy       : {rest_acc:.4f}')
print(f'   ➜ AD Accuracy       : {ad_acc:.4f}')
print(f'\n💾 Predictions saved to:')
print(f'   {pred_path}')
print('\n📌 Sample Predictions (First 15 Rows)')
print(df_pred.head(15).to_string(index=False))

print(' MODULE 12 COMPLETED: PER-SUBJECT PREDICTIONS EXPORTED')

In [ ]:
# @title Cell 16 - Cross-Dataset Evaluation
DATASETS = ["ADNI","OASIS_CROSS","OASIS_LONG", "KAGGLE_AD"]
results = []

for ds in DATASETS:
    print("\nRunning:", ds)
    X_cross_ds, X_long_ds, y_binary_ds, rid_list_ds, feat_cols_ds = load_dataset(ds)

    # Requirement 1: Subject-level split
    unique_ids = np.unique(rid_list_ds)

    # Map RIDs to their corresponding labels for stratification
    # Create a temporary DataFrame to easily map RIDs to labels and ensure unique RIDs
    temp_df = pd.DataFrame({'RID': rid_list_ds, 'label': y_binary_ds}).drop_duplicates(subset=['RID'])
    rid_to_label = temp_df.set_index('RID')['label']
    unique_ids_labels = rid_to_label.loc[unique_ids].values

    # Stratify the split of unique_ids based on their labels
    train_ids, test_ids, _, _ = train_test_split(
        unique_ids, unique_ids_labels, test_size=0.2, random_state=42, stratify=unique_ids_labels
    )

    train_mask = np.isin(rid_list_ds, train_ids)
    test_mask = np.isin(rid_list_ds, test_ids)

    X_tr_ds, X_te_ds = X_cross_ds[train_mask], X_cross_ds[test_mask]
    Xl_tr_ds, Xl_te_ds = X_long_ds[train_mask], X_long_ds[test_mask]
    y_tr_ds, y_te_ds = y_binary_ds[train_mask], y_binary_ds[test_mask]

    model_ds = TGSALPipeline()
    model_ds.fit(X_tr_ds, Xl_tr_ds, y_tr_ds, feat_cols_ds)

    y_pred_ds = model_ds.predict(X_te_ds, Xl_te_ds)
    y_proba_ds = model_ds.predict_proba(X_te_ds, Xl_te_ds)[:,1]

    results.append([ds, accuracy_score(y_te_ds, y_pred_ds), f1_score(y_te_ds, y_pred_ds), roc_auc_score(y_te_ds, y_proba_ds)])

df_results = pd.DataFrame(results, columns=["Dataset","Accuracy","F1","AUC"])
display(df_results)

In [ ]:
#  @title Cell 17 - Generate and Save All Figures

from sklearn.metrics import confusion_matrix, roc_curve, auc

def generate_all_figures(
    model,
    Xl_tr,
    y_tr,
    y_te,
    y_pred,
    y_proba,
    df_bio,
    OUTPUT_DIR,
    TIMEPOINTS
):

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("\n📊 Generating result figures...\n")

    # ----------------------------
    # 1. CONFUSION MATRIX
    # ----------------------------
    plt.figure(figsize=(5,4))

    sns.heatmap(
        confusion_matrix(y_te, y_pred),
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Rest','AD'],
        yticklabels=['Rest','AD'],
        annot_kws={'size':12,'weight':'bold'}
    )

    plt.title("Confusion Matrix (Rest vs AD)")
    plt.xlabel("Predicted")
    plt.ylabel("True Label")

    path=os.path.join(OUTPUT_DIR,"confusion_matrix.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()


    # ----------------------------
    # 2. FEATURE REDUCTION
    # ----------------------------
    plt.figure(figsize=(5,4))

    labels=[
        'Original\n(all)',
        'Stage 1\nLASSO',
        'Stage 2\nGroup',
        'Stage 3\nAdditive',
        'Stage 4\nTemporal'
    ]

    counts=[
        len(model._clean_names),
        int(model.s1.mask_.sum()),
        int(model._union_mask.sum()),
        int(len(model.s3.active_feat_)),
        int(model.s4.temp_mask_.sum())
    ]

    colors=['#607D8B','#2196F3','#9C27B0','#FF5722','#009688']

    bars=plt.bar(labels,counts,color=colors)

    for bar,val in zip(bars,counts):
        plt.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+3,
                 str(val),
                 ha='center',
                 fontweight='bold')

    plt.ylabel("Feature Count")
    plt.title("Stage-wise Feature Reduction")

    path=os.path.join(OUTPUT_DIR,"feature_reduction.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()


    # ----------------------------
    # 3. BIOMARKERS
    # ----------------------------
    plt.figure(figsize=(7,5))

    top = df_bio.head(15)

    plt.barh(
        range(len(top)),
        top["Importance"].values[::-1],
        color="#F44336"
    )

    labels = [
        f"{f[:18]} ({r})"
        for f, r in zip(
            top["Feature"].values[::-1],
            top["Brain_Region"].values[::-1]
        )
    ]

    plt.yticks(range(len(top)), labels, fontsize=8)
    plt.xlabel("Importance Score")
    plt.title("Top 15 Biomarkers with Brain Regions")

    path=os.path.join(OUTPUT_DIR,"top_biomarkers.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()


    # ----------------------------
    # 4. ROC CURVE
    # ----------------------------
    plt.figure(figsize=(5,4))

    fpr,tpr,_ = roc_curve(y_te,y_proba[:,1])
    roc_auc = auc(fpr,tpr)

    plt.plot(fpr,tpr,color='red',lw=2,
             label=f'AD vs Rest  AUC={roc_auc:.3f}')

    plt.plot([0,1],[0,1],'k--')

    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("ROC Curve (AD vs Rest)")
    plt.legend()

    path=os.path.join(OUTPUT_DIR,"roc_curve.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()


    # ----------------------------
    # 5. TEMPORAL PROGRESSION
    # ----------------------------
    plt.figure(figsize=(5,4))

    hi = next(
        (i for i,nm in enumerate(model._clean_names)
         if "HIPP" in nm.upper()),
        0
    )

    for label,color,name in zip(
        [0,1],
        ['#4CAF50','#F44336'],
        ['Rest','AD']
    ):
        mask = y_tr == label
        s = Xl_tr[mask,:,hi]

        mean = np.nanmean(s,axis=0)
        std  = np.nanstd(s,axis=0)

        plt.plot(range(len(TIMEPOINTS)), mean,
                 'o-', color=color, label=name)

        plt.fill_between(range(len(TIMEPOINTS)),
                         mean-std, mean+std,
                         color=color, alpha=0.12)

    plt.xticks(range(len(TIMEPOINTS)),TIMEPOINTS)

    feature=model._clean_names[hi]

    plt.xlabel("Timepoint")
    plt.ylabel("Value (norm.)")
    plt.title(f"Temporal Progression of {feature}")

    plt.legend()

    path=os.path.join(OUTPUT_DIR,"temporal_progression.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()


    # ----------------------------
    # 6. REGION IMPORTANCE
    # ----------------------------
    plt.figure(figsize=(6,5))

    region_imp = (
        df_bio.groupby("Brain_Region")["Importance"]
        .sum()
        .sort_values()
    )

    colors = plt.cm.Reds(np.linspace(0.4,0.9,len(region_imp)))

    region_imp.plot(kind='barh', color=colors)

    for i,v in enumerate(region_imp):
        plt.text(v+0.001,i,f"{v:.3f}",va='center')

    plt.xlabel("Total Importance Score")
    plt.title("Brain Region Importance")

    plt.tight_layout()

    path=os.path.join(OUTPUT_DIR,"region_importance.png")
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()

    print("\n✅ All figures saved in:", OUTPUT_DIR)

# --- Start: Code added to re-initialize dependencies if kernel state is lost ---
# This block is added to ensure essential variables are defined
# if the kernel was reset or preceding cells were not executed correctly.
# It assumes pipeline classes (Preprocessor, TGSALPipeline etc.) and data loading
# (X_cross, X_long, y_binary, rid_list, feat_cols) are already defined and available
# from previous, successfully executed cells.

# Re-load dataset if variables are missing (from Cell 4)
if 'X_cross' not in locals() or 'X_long' not in locals() or 'y_binary' not in locals() or 'rid_list' not in locals() or 'feat_cols' not in locals():
    print("Re-loading dataset due to missing data variables.")
    # Assuming DATASET_NAME is defined from Cell 4
    X_cross, X_long, y_binary, rid_list, feat_cols = load_dataset(DATASET_NAME)

# Re-run the data split and model training/prediction (from Cell 11/12)
if 'model' not in locals() or 'y_pred' not in locals() or 'y_proba' not in locals():
    print("Re-running model training and prediction due to missing variables.")
    # Assuming X_cross, X_long, y_binary, rid_list, feat_cols are defined from Cell 4
    # Assuming TGSALPipeline class is defined from Cell 10
    from sklearn.model_selection import train_test_split
    X_tr, X_te, Xl_tr, Xl_te, y_tr, y_te, rid_tr, rid_te = train_test_split(
        X_cross, X_long, y_binary, rid_list,
        test_size=0.2,
        stratify=y_binary,
        random_state=42
    )
    model = TGSALPipeline()
    model.fit(X_tr, Xl_tr, y_tr, feat_cols)
    y_pred  = model.predict(X_te, Xl_te)
    y_proba = model.predict_proba(X_te, Xl_te)

# Re-generate biomarker dataframe (from Cell 14)
if 'df_bio' not in locals():
    print("Re-generating biomarker dataframe.")
    # Assuming get_biomarker_df function is defined from Cell 14
    df_bio = get_biomarker_df(model)

# Ensure OUTPUT_DIR is defined (from Cell 15 or 18)
if 'OUTPUT_DIR' not in locals():
    print("Re-defining OUTPUT_DIR.")
    import os
    OUTPUT_DIR = "/content/output"
    os.makedirs(OUTPUT_DIR, exist_ok=True) # Ensure directory exists
# --- End: Code added to re-initialize dependencies ---

# Define TIMEPOINTS (adjust as needed if your dataset has different time labels)
TIMEPOINTS = ['T0', 'T1', 'T2', 'T3']

generate_all_figures(
    model=model,
    Xl_tr=Xl_tr,
    y_tr=y_tr,
    y_te=y_te,
    y_pred=y_pred,
    y_proba=y_proba,
    df_bio=df_bio,
    OUTPUT_DIR=OUTPUT_DIR,
    TIMEPOINTS=TIMEPOINTS
)

print('\n MODULE 13 COMPLETED: FIGURES GENERATED')

In [ ]:
# @title  Cell 18 — Final Summary
print('='*65)
print('  FINAL SUMMARY — TGSAL on Real ADNI Data')
print('='*65)
print(f'  Total subjects : {len(y_binary)}')
print(f'  Rest (CN+MCI)  : {(y_binary==0).sum()}')
print(f'  AD             : {(y_binary==1).sum()}')
print()
print(f'  Algorithm : Temporal Group Sparse Additive Learning (TGSAL)')
print(f'  Stage 1 LASSO            ->  {model.s1.mask_.sum()} features')
print(f'  Stage 2 Group LASSO      ->  {model._union_mask.sum()} features (union)')
print(f'  Stage 3 Sparse Additive  ->  {len(model.s3.active_feat_)} nonlinear features')
print(f'  Stage 4 Temporal SR      ->  {model.s4.temp_mask_.sum()} progressing features')
print()
print(f'  Test Accuracy  : {acc:.4f}')
print(f'  Weighted F1    : {f1:.4f}')
print(f'  ROC-AUC        : {auc_score:.4f}')
print(f'  CV Accuracy    : {np.mean(cv_acc):.4f} +/- {np.std(cv_acc):.4f}')
print()
print('  Top 10 Biomarkers:')
for _, row in df_bio.head(10).iterrows():
    print(f'    #{int(row["Rank"]):>2}  {row["Feature"]:<30}  '
          f'{row["Brain_Region"]:<22}  imp={row["Importance"]:.4f}')
print()
print(f'  Outputs -> {OUTPUT_DIR}')
print('='*65)